# 01 — Exploration des données NYC Yellow Taxi

**Objectif** : comprendre les données *avant* d'écrire `transform.py`, et obtenir les **observations chiffrées** qui justifient chaque seuil de nettoyage (percentiles, distributions, schémas de valeurs manquantes).

Ce notebook est **séparé du code de production** : le pipeline ne dépend jamais de lui.

**Deux sources possibles** (paramètre `SOURCE` ci-dessous) :
- `"raw"` : lit les pages brutes produites par l'extraction. Lance d'abord :
  `python -m pipeline --year 2023 --month 1 --steps extract`
- `"api"` : interroge directement l'API pour un échantillon de `SAMPLE_LIMIT` lignes (pratique pour un premier coup d'œil).

> ⚠️ Avant chaque commit : *Kernel → Restart & Clear All Outputs* (ou `jupyter nbconvert --clear-output --inplace notebooks/01_exploration.ipynb`).

In [ ]:
# --- Imports et accès au package du projet ---------------------------------
import sys
from pathlib import Path

# Le notebook vit dans notebooks/ : on ajoute la racine du projet au chemin
# Python pour pouvoir importer le package `pipeline` (config, io_utils, extract).
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline.config import load_config, make_period, resolve_path, load_token
from pipeline.io_utils import read_json
from pipeline import extract

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Racine du projet :", PROJECT_ROOT)

In [ ]:
# --- Paramètres de l'exploration --------------------------------------------
YEAR, MONTH = 2023, 1

SOURCE = "raw"          # "raw" (pages extraites par le pipeline) ou "api" (échantillon direct)

# Mode "raw" : nombre de pages à charger. None = tout le mois (~3 M lignes, plusieurs Go de RAM
# pendant le chargement). Un entier = échantillon aléatoire de pages (exploration plus rapide).
MAX_PAGES = None

# Mode "api" : paramètre $limit = nombre de lignes demandées en une seule requête.
# Sans $limit, l'API renverrait seulement 1000 lignes.
SAMPLE_LIMIT = 50_000

RANDOM_STATE = 42       # reproductibilité du tirage des pages

In [ ]:
# --- Configuration commune avec le pipeline ----------------------------------
cfg = load_config(PROJECT_ROOT / "config.yaml")
period = make_period(YEAR, MONTH)
raw_dir = resolve_path(cfg, "raw_dir", period)
SCHEMA = cfg["schema"]      # typage attendu de chaque colonne (config.yaml)

print("Mois :", period.label, "| bornes :", period.start, "->", period.end)
print("Pages brutes :", raw_dir)

## 0. Chargement en DataFrame pandas

L'API renvoie **toutes les valeurs sous forme de chaînes** (`"1.0"`, `"9.3"`…). On conserve donc :
- `df_raw` : un extrait **non typé** (chaînes), pour constater le problème ;
- `df` : le DataFrame **typé** sur lequel porte toute l'exploration.

Le typage est fait page par page avant concaténation, ce qui divise fortement la mémoire utilisée.

In [ ]:
def cast_types(frame: pd.DataFrame, schema: dict) -> tuple[pd.DataFrame, dict]:
    """
    Convertit les colonnes selon le schéma et COMPTE les conversions ratées
    (valeur présente mais illisible -> NaN). errors="coerce" évite un plantage,
    mais une conversion ratée non comptée serait une anomalie silencieuse.
    """
    frame = frame.copy()
    failures = {}

    def track(col, before_na):
        failures[col] = int(frame[col].isna().sum() - before_na)

    for col in schema["datetime"]:
        if col in frame:
            na = frame[col].isna().sum()
            frame[col] = pd.to_datetime(frame[col], format="%Y-%m-%dT%H:%M:%S.%f", errors="coerce")
            track(col, na)
    for col in schema["float"]:
        if col in frame:
            na = frame[col].isna().sum()
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
            track(col, na)
    for col in schema["integer"]:
        if col in frame:
            na = frame[col].isna().sum()
            num = pd.to_numeric(frame[col], errors="coerce")
            # "1.0" -> 1 ; on garde trace des valeurs à partie décimale réelle (ex. 1.5)
            failures[f"{col}__non_entier"] = int((num.notna() & (num % 1 != 0)).sum())
            frame[col] = num.round().astype("Int64")   # Int64 = entier qui accepte les NaN
            track(col, na)
    for col in schema["category"]:
        if col in frame:
            frame[col] = frame[col].astype("category")
    return frame, failures


def load_from_raw(raw_dir: Path, max_pages, seed):
    """Charge les pages .json(.gz) écrites par l'étape d'extraction."""
    pages = sorted(p for p in raw_dir.glob("page_*.json*"))
    if not pages:
        raise FileNotFoundError(
            f"Aucune page dans {raw_dir}. Lance d'abord : "
            f"python -m pipeline --year {YEAR} --month {MONTH} --steps extract  (ou SOURCE='api')"
        )
    if max_pages is not None and max_pages < len(pages):
        random.Random(seed).shuffle(pages)          # pages tirées au hasard, pas les premières
        pages = sorted(pages[:max_pages])
    print(f"Chargement de {len(pages)} page(s)...")

    typed, total_fail, first_raw = [], {}, None
    for i, p in enumerate(pages):
        page_df = pd.DataFrame(read_json(p))
        if first_raw is None:
            first_raw = page_df.head(1000)          # extrait brut (chaînes) conservé
        page_df, fail = cast_types(page_df, SCHEMA)
        typed.append(page_df)
        for k, v in fail.items():
            total_fail[k] = total_fail.get(k, 0) + v
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(pages)} pages")
    return first_raw, pd.concat(typed, ignore_index=True), total_fail


def load_from_api(limit: int):
    """Échantillon direct via l'API : $limit lignes, triées sur :id, filtrées sur le mois."""
    session = extract.build_session(load_token(cfg))
    params = {
        "$select": "*, :id",
        "$where": extract.build_where(cfg, period),
        "$order": cfg["api"]["order_by"],
        "$limit": limit,                     # <- taille de l'échantillon
    }
    rows = extract.request_with_retry(session, extract.endpoint(cfg), params, cfg["network"])
    raw = pd.DataFrame(rows)
    typed, fail = cast_types(raw, SCHEMA)
    print("Attention : les premières lignes par :id ne sont PAS un échantillon aléatoire.")
    return raw.head(1000), typed, fail


if SOURCE == "raw":
    df_raw, df, conversion_failures = load_from_raw(raw_dir, MAX_PAGES, RANDOM_STATE)
elif SOURCE == "api":
    df_raw, df, conversion_failures = load_from_api(SAMPLE_LIMIT)
else:
    raise ValueError("SOURCE doit valoir 'raw' ou 'api'")

print(f"df : {len(df):,} lignes x {df.shape[1]} colonnes | "
      f"{df.memory_usage(deep=True).sum() / 1e6:,.0f} Mo en mémoire")
df.head()

## 1. Types : ce que renvoie vraiment l'API

Questions : le type Python réel correspond-il à ce qu'on attend en lisant le JSON ? Une valeur qui *ressemble* à un nombre est-elle stockée comme un nombre ?

In [ ]:
# Type Python des valeurs dans le JSON brut (première ligne)
first = df_raw.iloc[0].to_dict()
print({k: type(v).__name__ for k, v in first.items()})

# dtypes AVANT typage (tout en object = chaînes) vs APRÈS
pd.DataFrame({"brut (API)": df_raw.dtypes.astype(str), "typé": df.dtypes.astype(str)})

In [ ]:
# Conversions ratées (valeur présente mais illisible) et valeurs non entières
# dans les colonnes censées être entières. À reporter dans le rapport qualité.
pd.Series(conversion_failures, name="nb_lignes").sort_values(ascending=False)

## 2. Valeurs manquantes : aléatoires ou structurées ?

In [ ]:
na_rate = df.isna().mean().sort_values(ascending=False)
na_rate[na_rate > 0].to_frame("taux_NaN")

In [ ]:
# Les NaN apparaissent-ils ENSEMBLE sur les mêmes lignes ?
na_cols = na_rate[na_rate > 0].index.tolist()
if na_cols:
    display(df[na_cols].isna().value_counts().to_frame("nb_lignes"))

In [ ]:
# Avec quelles autres colonnes sont-ils liés ?
if na_cols:
    missing = df[na_cols[0]].isna()
    print(f"Référence : NaN dans '{na_cols[0]}' ({missing.sum():,} lignes)\n")
    for col in ["payment_type", "vendorid"]:
        display(pd.crosstab(df[col], missing, margins=True,
                            rownames=[col], colnames=[f"{na_cols[0]} manquant"]))
    # Ces lignes ont-elles des montants valides ? (=> trajets réels à conserver ?)
    display(df.loc[missing, ["trip_distance", "fare_amount", "total_amount"]].describe())

## 3. Dates et durées

In [ ]:
start, end = pd.Timestamp(period.start), pd.Timestamp(period.end)
df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

checks = {
    "pickup hors du mois": (~df["tpep_pickup_datetime"].between(start, end, inclusive="left")).sum(),
    "dropoff avant pickup (durée < 0)": (df["duration_min"] < 0).sum(),
    "durée = 0": (df["duration_min"] == 0).sum(),
    "dropoff manquant": df["tpep_dropoff_datetime"].isna().sum(),
}
pd.Series(checks, name="nb_lignes").to_frame()

## 4. Distributions : distances, durées, montants

Les seuils de nettoyage doivent venir de **ces percentiles**, pas d'une valeur qui « semble raisonnable ». Regarde surtout les queues (p0.1 / p99.9) et l'écart entre p99 et max.

In [ ]:
PCTS = [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
num_cols = ["trip_distance", "duration_min", "fare_amount", "tip_amount",
            "tolls_amount", "total_amount", "extra"]
df[num_cols].describe(percentiles=PCTS).T

In [ ]:
# Valeurs nulles et négatives : combien, et pour quels types de paiement ?
summary = pd.DataFrame({
    "nb_negatifs": (df[num_cols] < 0).sum(),
    "nb_zeros": (df[num_cols] == 0).sum(),
})
display(summary)
display(pd.crosstab(df["payment_type"], df["total_amount"] < 0,
                    colnames=["total_amount < 0"], margins=True))

In [ ]:
# Histogrammes en échelle log (les queues sont invisibles en échelle linéaire).
# Affichage tronqué au p99.9 pour rester lisible ; le max est indiqué en titre.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["trip_distance", "duration_min", "total_amount"]):
    s = df[col].dropna()
    hi = s.quantile(0.999)
    ax.hist(s[(s >= 0) & (s <= hi)], bins=100)
    ax.set_yscale("log")
    ax.set_title(f"{col} (≤ p99.9 = {hi:,.1f} ; max = {s.max():,.1f})")
plt.tight_layout()
plt.show()

In [ ]:
# Vitesse moyenne implicite : un bon révélateur d'incohérence distance/durée
# (ex. 50 miles en 2 minutes). Utile pour justifier un seuil de distance ou de durée.
valid = (df["duration_min"] > 0) & (df["trip_distance"] > 0)
speed_mph = df.loc[valid, "trip_distance"] / (df.loc[valid, "duration_min"] / 60)
speed_mph.describe(percentiles=PCTS).to_frame("vitesse_mph")

### 4 bis. Seuils candidats : combien de lignes chaque seuil retirerait-il ?

In [ ]:
# Combien de lignes chaque seuil candidat retirerait-il ?
speed = df["trip_distance"] / (df["duration_min"] / 60)
candidates = {
    "trip_distance >": (df["trip_distance"], [30, 50, 100, 200]),
    "duration_min >":  (df["duration_min"], [180, 240, 300, 600]),
    "vitesse_mph >":   (speed.where(df["duration_min"] > 0), [50, 60, 80, 100]),
    "total_amount >":  (df["total_amount"], [200, 300, 500]),
}
rows = []
for name, (s, seuils) in candidates.items():
    for t in seuils:
        n = int((s > t).sum())
        rows.append({"règle": f"{name} {t}", "lignes": n, "%": f"{n / len(df):.3%}"})
display(pd.DataFrame(rows))

print("p99.99 :", df[["trip_distance", "duration_min", "total_amount"]].quantile(0.9999).round(2).to_dict())
print("Distance nulle avec montant > 0 :", int(((df["trip_distance"] == 0) & (df["total_amount"] > 0)).sum()))

## 5. Cohérence financière : `total_amount` = somme des composantes ?

Si l'écart n'est pas nul, est-ce une erreur à corriger ou un **signal à documenter** ? Hypothèse à tester : chez `vendorid = 1`, le champ `extra` inclurait déjà certaines surcharges (congestion, aéroport).

In [ ]:
COMPONENTS = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
              "improvement_surcharge", "congestion_surcharge", "airport_fee"]
df["ecart_total"] = (df["total_amount"] - df[COMPONENTS].sum(axis=1, min_count=1)).round(2)

# Écarts les plus fréquents, par fournisseur
df.groupby("vendorid")["ecart_total"].value_counts().groupby(level=0).head(8).to_frame("nb_lignes")

In [ ]:
# Test de l'hypothèse : l'écart correspond-il exactement à une surcharge ?
d = df[df["ecart_total"] != 0]
explained = pd.DataFrame({
    "ecart = -congestion": np.isclose(d["ecart_total"], -d["congestion_surcharge"]),
    "ecart = -airport_fee": np.isclose(d["ecart_total"], -d["airport_fee"]),
    "ecart = -(congestion + airport)": np.isclose(d["ecart_total"], -(d["congestion_surcharge"] + d["airport_fee"])),
}, index=d.index)
explained["vendorid"] = d["vendorid"]
print(f"Lignes avec écart non nul : {len(d):,} ({len(d) / len(df):.2%})")
explained.groupby("vendorid").sum()

## 6. Codes catégoriels

`vendorid`, `payment_type`, `ratecodeid`, `pulocationid`/`dolocationid` sont des **codes**, pas des quantités : on regarde leurs modalités, jamais leur moyenne. À comparer avec le dictionnaire officiel TLC.

In [ ]:
for col in ["vendorid", "payment_type", "ratecodeid", "passenger_count", "store_and_fwd_flag"]:
    display(df[col].value_counts(dropna=False).to_frame(col))

In [ ]:
# Zones 264 / 265 = zones inconnues dans la table des zones TLC
for col in ["pulocationid", "dolocationid"]:
    print(col, "->", df[col].isin([264, 265]).sum(), "lignes en zone inconnue")

# passenger_count = 0 : par fournisseur ?
display(pd.crosstab(df["vendorid"], df["passenger_count"] == 0, colnames=["passenger_count == 0"]))

## 7. Synthèse : décisions à reporter dans `config.yaml`, `transform.py` et le README

Remplis ce tableau avec **tes chiffres** avant d'écrire `transform.py`.

| Règle | Observation chiffrée | Seuil retenu | Décision (supprimer / conserver + marquer) |
|---|---|---|---|
| Distance max | p99.9 = … ; vitesse implicite au-delà = … | … | … |
| Distance nulle | … lignes ; dont … avec montant > 0 | … | … |
| Durée min / max | p0.1 = … ; p99.9 = … | … | … |
| Durée négative | … lignes | — | … |
| Pickup hors du mois | … lignes | — | … |
| Montants négatifs | … lignes ; payment_type majoritaire = … | — | … |
| Métadonnées manquantes ensemble | … lignes ; lien avec … | — | … |
| Écart `total_amount` (vendor 1) | … % des lignes ; = surcharge dans … % des cas | — | … |
| `passenger_count = 0` | … lignes ; vendor … | — | … |
| Zones 264/265 | … lignes | — | … |